# PROB IP102 Continual Learning Evaluation & Training Pipeline

Notebook này cung cấp quy trình (pipeline) hoàn chỉnh để chạy đánh giá và huấn luyện mô hình **PROB** (Probabilistic Open-World Object Detection) với bộ dữ liệu sâu bệnh **IP102** trên **Kaggle** sử dụng **2 GPU T4 (DDP)**.

### Hướng dẫn chuẩn bị trước khi chạy:
1. Đảm bảo Notebook này được cấu hình chạy trên **Accelerator: GPU T4 x2** (ở panel Settings phía bên phải).
2. Đảm bảo **Internet** được bật (Internet on).
3. Thêm các Dataset sau vào Notebook:
   - Tập dữ liệu ảnh IP102 (ví dụ: `rtlmhjbn/ip02-dataset` hoặc tương đương).
   - Tập nhãn COCO của IP102 (ví dụ: `eljazouly/ip102-coco-annotations` hoặc tương đương).
   - Thư mục chứa các checkpoint pretrain tác giả hoặc của bạn (ví dụ: `prob-checkpoints` chứa `t1.pth`, `t2.pth`, `t3.pth`, `t4.pth`).

## Bước 1: Clone Repository và Cài đặt thư viện

In [ ]:
# Clone repo chứa mã nguồn tùy chỉnh đã được cấu hình lớp IP102
!git clone https://github.com/nta2112/Prob-ann-custom.git
%cd Prob-ann-custom

In [ ]:
# Cài đặt các thư viện yêu cầu
!pip install -r requirements.txt
!pip install pycocotools einops

## Bước 2: Thiết lập dữ liệu IP102 (Chuyển đổi sang định dạng VOC XML)

Mã nguồn PROB sử dụng dataloader định dạng VOC XML. Lệnh dưới đây chạy script `setup_ip102.py` để quét ảnh, tự động vá file cấu hình lớp `open_world.py` và chuyển nhãn từ COCO JSON sang VOC XML.

In [ ]:
# LƯU Ý: Hãy kiểm tra và điều chỉnh đường dẫn bên dưới nếu tên Dataset của bạn khác
JSON_DIR = "/kaggle/input/ip102-coco-annotations/coco_annotations"
IMAGE_DIR = "/kaggle/input/ip02-dataset"
OUTPUT_DIR = "/kaggle/working/datasets/IP102"

!python tools/setup_ip102.py \
    --json_dir {JSON_DIR} \
    --image_dir {IMAGE_DIR} \
    --output_dir {OUTPUT_DIR}

## Bước 3: Chạy Đánh giá (Evaluation) sử dụng 2 GPU

Chạy kịch bản `run_eval_kaggle.sh` để thực hiện đánh giá Học liên tục trên cả 4 task. 
- Tham số thứ nhất: Đường dẫn thư mục chứa checkpoint (`t1.pth`, `t2.pth`...).
- Tham số thứ hai: Batch size trên mỗi GPU (Khuyến nghị: `16` cho đánh giá để chạy nhanh nhất).

In [ ]:
# LƯU Ý: Hãy sửa '/kaggle/input/prob-checkpoints' thành đường dẫn chứa checkpoint của bạn trên Kaggle
!bash tools/run_eval_kaggle.sh /kaggle/input/prob-checkpoints 16

## Bước 4: Huấn luyện thực tế (Training) Học liên tục trên IP102 (Tùy chọn)

Nếu bạn muốn chạy huấn luyện thực tế thay vì chỉ đánh giá, bạn có thể thực hiện chạy tuần tự qua các task. Dưới đây là lệnh mẫu chạy huấn luyện **Task 1** (sử dụng 2 GPU, batch size = 4 mỗi GPU để tránh tràn bộ nhớ VRAM).

In [ ]:
# Chạy huấn luyện Task 1 với 27 lớp đầu tiên của IP102
!python -m torch.distributed.run --nproc_per_node=2 --master_port=29509 main_open_world.py \
    --output_dir "/kaggle/working/exps/IP102/t1" \
    --dataset IP102 \
    --num_classes 103 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 27 \
    --train_set "owod_t1_train" \
    --test_set "owod_all_task_test" \
    --model_type "prob" \
    --obj_loss_coef 8e-4 \
    --obj_temp 1.3 \
    --epochs 41 \
    --batch_size 4 \
    --num_workers 2 \
    --data_root "/kaggle/working/datasets/IP102"